# Scaled reproduction of Figures 2 and 3

This notebook runs the comparison and LoRA fine-tuning sweeps for Qwen2.5-0.5B-Instruct. It generates plots and raw run outputs in Kaggle; output images are not part of the source repository. Enable Kaggle Internet and a GPU. A Hugging Face read token with access to LMSYS-Chat-1M is required.

In [ ]:
# Setup: choose the source branch (use main after the PR is merged).
import os, subprocess, sys, uuid
REPO_REF = 'codex/reproduction-code-no-output-images'
REPO_URL = 'https://github.com/hiraaziz/trustworthy-AI.git'
repo = '/kaggle/working/trustworthy-AI'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, repo], check=True)
else:
    subprocess.run(['git', '-C', repo, 'fetch', 'origin', REPO_REF, '--depth', '1'], check=True)
    subprocess.run(['git', '-C', repo, 'checkout', 'FETCH_HEAD'], check=True)
project = os.path.join(repo, 'reproducibility', 'Figure2-3-Scaled-Comparison', 'source', 'track-llm-apis')
run_id = 'qwen05-fig2-3-' + uuid.uuid4().hex[:8]
os.environ['TRACKLLM__DATA_DIR'] = '/kaggle/working/' + run_id
os.environ['B3IT_REPRO_ROOT'] = '/kaggle/working'
os.environ['B3IT_REPRO_MODEL_DIR'] = '/kaggle/working/models/Qwen2.5-0.5B-Instruct'
os.environ['B3IT_REPRO'] = '1'
os.environ['B3IT_REPRO_FINETUNE_STEPS'] = '1,2,4,8,16,32,64,128,256,512'
os.environ['B3IT_REPRO_FINETUNE_MAX_LENGTH'] = '128'
os.environ['B3IT_REPRO_SKIP_FULL_FINETUNE'] = '1'
os.environ['B3IT_REPRO_SKIP_LORA'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print('Run:', run_id, '| source:', project)

## Install the pinned Kaggle runtime
Run this once per fresh Kaggle session. If Kaggle asks for a kernel restart, choose **Restart session**. After restart, rerun the setup cell above, then run the remaining cells in order.

In [ ]:
%pip install -q uv
!uv pip install --system --reinstall torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!uv pip install --system --reinstall vllm==0.10.2 --torch-backend=cu128 --extra-index-url https://wheels.vllm.ai/0.10.2/
%pip install --no-cache-dir --force-reinstall --no-deps 'huggingface_hub==0.36.2' 'transformers==4.56.1' 'tokenizers==0.22.1'
%pip uninstall -y torchao
%pip install -e "{project}" --no-deps

In [ ]:
# Verify the repaired runtime before starting a long run.
import torch, transformers, tokenizers, huggingface_hub, vllm
print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'available', torch.cuda.is_available())
print('transformers', transformers.__version__, 'tokenizers', tokenizers.__version__, 'hub', huggingface_hub.__version__, 'vLLM', vllm.__version__)
if not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU accelerator and restart the session.')
print('GPU count:', torch.cuda.device_count())
assert transformers.__version__ == '4.56.1'
assert tokenizers.__version__ == '0.22.1'
assert huggingface_hub.__version__ == '0.36.2'

In [ ]:
# Load Kaggle's HF_TOKEN secret and stream a filtered 512-example LoRA dataset.
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
from pathlib import Path
import json
token = UserSecretsClient().get_secret('HF_TOKEN')
if not token: raise RuntimeError('Add a Kaggle secret named HF_TOKEN with a Hugging Face read token.')
os.environ['HF_TOKEN'] = token
stream = load_dataset('lmsys/lmsys-chat-1m', split='train', streaming=True, token=token)
records = []
for row in stream:
    conv = row.get('conversation') or []
    moderation = row.get('openai_moderation') or []
    if row.get('model') != 'gpt-4' or row.get('redacted', True) or len(conv) < 2: continue
    if any(m.get('flagged', False) for m in moderation if isinstance(m, dict)): continue
    if conv[0].get('role') != 'user' or conv[1].get('role') != 'assistant': continue
    records.append({'conversation': conv[:2]})
    if len(records) == 512: break
if len(records) < 512: raise RuntimeError(f'Only found {len(records)} eligible records; check dataset access/schema.')
train_json = Path('/kaggle/working/lmsys_gpt4_first_turn_512.jsonl')
train_json.write_text('\n'.join(json.dumps(row, ensure_ascii=False) for row in records) + '\n', encoding='utf-8')
os.environ['B3IT_REPRO_FINETUNE_JSON'] = str(train_json)
print('Filtered training examples:', len(records), '| saved:', train_json)

In [ ]:
# Download the base model once into the Kaggle working directory.
from huggingface_hub import snapshot_download
model_dir = Path(os.environ['B3IT_REPRO_MODEL_DIR'])
snapshot_download(repo_id='Qwen/Qwen2.5-0.5B-Instruct', local_dir=str(model_dir), token=token)
print('Model ready:', model_dir)

## Run the reproduction
This runs the method comparison and LoRA update sweep from 1 to 512 examples/steps. It can take several hours. Kaggle sessions may stop when they idle or hit their time limit; the source runner reuses completed sampling output when the same data directory persists. Keep the notebook active and monitor the log.

In [ ]:
import subprocess, sys
runner = os.path.join(project, 'reproduction', 'run_scaled_reproduction.py')
subprocess.run([sys.executable, runner], cwd=project, env=os.environ.copy(), check=True)

In [ ]:
# Locate runtime-generated figures and make a downloadable archive; nothing is committed to Git.
from zipfile import ZipFile, ZIP_DEFLATED
data_dir = Path(os.environ['TRACKLLM__DATA_DIR'])
plots_dir = data_dir / 'plots' / 'paper'
print('Plots directory:', plots_dir, '| exists:', plots_dir.exists())
artifact = Path('/kaggle/working/' + run_id + '-results.zip')
with ZipFile(artifact, 'w', ZIP_DEFLATED) as zf:
    for p in data_dir.rglob('*'):
        if p.is_file(): zf.write(p, p.relative_to(data_dir.parent))
print('Download from Kaggle Output:', artifact)
if plots_dir.exists(): print('Generated plot files:', *[str(p) for p in plots_dir.rglob('*') if p.is_file()], sep='\n')